In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split


In [2]:
# Setup data pipeline for this notebook
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root="data", transform=data_transforms)
train_size = int(0.70 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_data, val_data, test_data = random_split(
    full_dataset, 
    [train_size, val_size, test_size], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"Data ready: {len(train_data)} train, {len(val_data)} val, {len(test_data)} test")


Data ready: 2951 train, 632 val, 634 test


In [3]:
# Load pretrained MobileNetV2 and freeze feature backbone
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)

# Freeze feature extractor layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace final classification layer for our 4 classes
model.classifier[1] = nn.Linear(in_features=1280, out_features=4)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\Nayera Alashry/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:06<00:00, 2.32MB/s]


Total parameters: 2,228,996
Trainable parameters: 5,124


In [4]:
# Loss function and optimizer for trainable parameters only
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam (lr=0.001, tracking 5,124 trainable weights)")


Loss function: CrossEntropyLoss
Optimizer: Adam (lr=0.001, tracking 5,124 trainable weights)


In [5]:
# Training MobileNetV2 model for 3 epochs
epochs = 3

for epoch in range(epochs):
    # Training phase
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    train_loss = running_loss / total
    train_acc = (correct / total) * 100
    
    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    val_loss = val_loss / val_total
    val_acc = (val_correct / val_total) * 100
    
    print(f"Epoch {epoch+1}/{epochs} -> Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")


Epoch 1/3 -> Train Loss: 0.8645, Acc: 68.42% | Val Loss: 0.6325, Acc: 73.10%
Epoch 2/3 -> Train Loss: 0.6119, Acc: 77.50% | Val Loss: 0.5121, Acc: 83.07%
Epoch 3/3 -> Train Loss: 0.5272, Acc: 81.46% | Val Loss: 0.4721, Acc: 84.34%


In [6]:
# Evaluate MobileNetV2 on the unseen Test set
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

final_test_loss = test_loss / test_total
final_test_acc = (test_correct / test_total) * 100

print(f"Final MobileNetV2 Test Loss: {final_test_loss:.4f} | Test Accuracy: {final_test_acc:.2f}%")


Final MobileNetV2 Test Loss: 0.4703 | Test Accuracy: 83.44%


In [7]:
# Save trained model weights for future inference and demo deployment
torch.save(model.state_dict(), "mobilenet_v2_eye_disease.pth")
print("Model weights successfully saved to mobilenet_v2_eye_disease.pth")


Model weights successfully saved to mobilenet_v2_eye_disease.pth
